# 09 - Modelos robustos con busqueda exhaustiva de hiperparametros

Este notebook aplica GridSearchCV con varios modelos sobre las features de fusion temprana y exporta predicciones JSON en predicciones/.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

In [ ]:
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd] + list(cwd.parents)
    for c in candidates:
        if (c / 'materials' / 'dataset_task3_exist2026').exists() and (c / 'notebooks_shiyi').exists():
            return c / 'notebooks_shiyi'
    if (cwd / 'artifacts').exists() and (cwd / 'entregables').exists():
        return cwd
    raise FileNotFoundError('No se pudo resolver notebooks_shiyi.')

PROJECT_ROOT = resolve_project_root()
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
PRED_DIR = PROJECT_ROOT / 'predicciones_finales'
WEIGHTS_DIR = PROJECT_ROOT / 'weights_hpo'
REPORTS_DIR = PROJECT_ROOT / 'reports_hpo'
for d in [PRED_DIR, WEIGHTS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

bundle = np.load(ARTIFACTS_DIR / 'fusion_features.npz', allow_pickle=True)
X_all = bundle['X_fusion'].astype(np.float32)
y_all = bundle['y'].astype(np.int64)
langs_all = bundle['langs'].astype(str)
sample_ids = bundle['sample_ids'].astype(str)
TEST_JSON_PATH = PROJECT_ROOT.parent / 'materials' / 'dataset_task3_exist2026' / 'test.json'
if not TEST_JSON_PATH.exists():
    raise FileNotFoundError(f'test.json not found: {TEST_JSON_PATH}')
with open(TEST_JSON_PATH, 'r', encoding='utf-8') as f:
    test_raw = json.load(f)
TEST_IDS = {str(k) for k in test_raw.keys()}
test_mask_all = np.array([sid in TEST_IDS for sid in sample_ids], dtype=bool)
            
print('X_all:', X_all.shape, '| labels validas:', int((y_all >= 0).sum()), '| test rows:', int(test_mask_all.sum()))

In [ ]:
def get_model_spaces():
    spaces = []

    svm_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(probability=True, class_weight='balanced', random_state=42))
    ])
    svm_grid = {
        'clf__C': [0.1, 1.0, 10.0, 30.0],
        'clf__gamma': ['scale', 0.01, 0.001],
        'clf__kernel': ['rbf']
    }
    spaces.append(('svm_rbf', svm_pipe, svm_grid))

    rf = RandomForestClassifier(class_weight='balanced_subsample', random_state=42, n_jobs=-1)
    rf_grid = {
        'n_estimators': [300, 600],
        'max_depth': [None, 20, 40],
        'min_samples_split': [2, 8],
        'min_samples_leaf': [1, 3]
    }
    spaces.append(('random_forest', rf, rf_grid))

    et = ExtraTreesClassifier(class_weight='balanced', random_state=42, n_jobs=-1)
    et_grid = {
        'n_estimators': [300, 600],
        'max_depth': [None, 20, 40],
        'min_samples_split': [2, 8],
        'min_samples_leaf': [1, 3]
    }
    spaces.append(('extra_trees', et, et_grid))

    hgb = HistGradientBoostingClassifier(random_state=42)
    hgb_grid = {
        'learning_rate': [0.03, 0.06, 0.1],
        'max_depth': [None, 8, 12],
        'max_leaf_nodes': [31, 63],
        'min_samples_leaf': [20, 40]
    }
    spaces.append(('hist_gb', hgb, hgb_grid))

    return spaces

In [ ]:
def export_json(sample_ids, pred_binary, output_path):
    label_map = {0: 'NO', 1: 'YES'}
    rows = [
        {'test_case': 'EXIST2026', 'id': str(sid), 'value': label_map[int(v)]}
        for sid, v in zip(sample_ids, pred_binary)
    ]
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(rows, f, ensure_ascii=False)
    print('Exportado:', output_path, '| filas:', len(rows))

def run_config(tag, train_langs):
    mask = np.isin(langs_all, train_langs) & (y_all >= 0)
    X_train = X_all[mask]
    y_train = y_all[mask]

    if len(np.unique(y_train)) < 2:
        raise RuntimeError(f'{tag}: subset sin dos clases.')

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    best_score = -1.0
    best_name = None
    best_model = None
    all_rows = []

    for model_name, estimator, param_grid in get_model_spaces():
        gs = GridSearchCV(
            estimator=estimator,
            param_grid=param_grid,
            scoring='f1_macro',
            cv=cv,
            n_jobs=-1,
            verbose=1,
            refit=True
        )
        gs.fit(X_train, y_train)

        row = {
            'config': tag,
            'model': model_name,
            'best_cv_f1_macro': float(gs.best_score_)
        }
        row.update({f'param_{k}': v for k, v in gs.best_params_.items()})
        all_rows.append(row)

        if gs.best_score_ > best_score:
            best_score = float(gs.best_score_)
            best_name = model_name
            best_model = gs.best_estimator_

    rep_df = pd.DataFrame(all_rows).sort_values('best_cv_f1_macro', ascending=False)
    rep_path = REPORTS_DIR / f'cv_results_{tag}.csv'
    rep_df.to_csv(rep_path, index=False)
    print('Reporte CV:', rep_path)

    best_model.fit(X_train, y_train)
    pred_train = best_model.predict(X_train)
    print(tag, 'best_model=', best_name, '| CV_f1=', best_score)
    print('F1 train:', f1_score(y_train, pred_train, average='macro'))
    print(classification_report(y_train, pred_train, digits=4))

    model_path = WEIGHTS_DIR / f'best_{tag}_{best_name}.joblib'
    joblib.dump(best_model, model_path)
    print('Modelo guardado:', model_path)

    infer_mask = test_mask_all & np.isin(langs_all, train_langs)
    if not infer_mask.any():
        raise RuntimeError(f'{tag}: no test rows for selected langs')
    pred_test = best_model.predict(X_all[infer_mask])
    json_path = PRED_DIR / f'BeingChillingWeWillWin_HPO_{tag}.json'
    export_json(sample_ids[infer_mask], pred_test, json_path)

run_config('ES', ['es'])
run_config('EN', ['en'])
run_config('ES_EN', ['es', 'en'])
